# Tokenizer From Scratch (Byte-Level BPE)

> Goal: build a production-style tokenizer with train, encode, decode, and serialization support.
> Dataset for initial training: `wizard_of_oz.txt`.
> This tokenizer is designed to plug directly into your future embedding and attention notebooks.

In [7]:
from __future__ import annotations

import json
import re
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np

In [8]:
@dataclass
class TokenizerConfig:
    vocab_size: int = 2000
    min_pair_freq: int = 2
    special_tokens: Tuple[str, ...] = ("<pad>", "<bos>", "<eos>", "<unk>")


class BytePairTokenizer:
    """
    Byte-level BPE tokenizer with reversible UTF-8 decode and JSON save/load.
    Design goals: deterministic training, compact model files, and fast encode/decode.
    """

    _word_re = re.compile(r"\s+|[^\s]+")

    def __init__(self, config: TokenizerConfig | None = None):
        self.config = config or TokenizerConfig()
        self.base_vocab_size = 256

        self.special_tokens = list(self.config.special_tokens)
        self.special_to_id: Dict[str, int] = {}
        self.id_to_special: Dict[int, str] = {}

        self.merges: Dict[Tuple[int, int], int] = {}
        self.merges_rank: Dict[Tuple[int, int], int] = {}
        self.token_to_bytes: Dict[int, bytes] = {i: bytes([i]) for i in range(self.base_vocab_size)}

        self._init_special_tokens()

    def _init_special_tokens(self) -> None:
        start = self.base_vocab_size
        for i, tok in enumerate(self.special_tokens):
            tid = start + i
            self.special_to_id[tok] = tid
            self.id_to_special[tid] = tok
            self.token_to_bytes[tid] = tok.encode("utf-8")

    @property
    def vocab_size(self) -> int:
        return len(self.token_to_bytes)

    def _merge_sequence(self, seq: Tuple[int, ...], pair: Tuple[int, int], new_id: int) -> Tuple[int, ...]:
        if len(seq) < 2:
            return seq

        out = []
        i = 0
        a, b = pair
        n = len(seq)
        while i < n:
            if i < n - 1 and seq[i] == a and seq[i + 1] == b:
                out.append(new_id)
                i += 2
            else:
                out.append(seq[i])
                i += 1
        return tuple(out)

    def _pretokenize(self, text: str) -> List[str]:
        return self._word_re.findall(text)

    def train(self, text: str, verbose: bool = True) -> None:
        if not text:
            raise ValueError("Cannot train tokenizer on empty text.")

        chunks = self._pretokenize(text)
        word_freqs = Counter(tuple(chunk.encode("utf-8")) for chunk in chunks)

        max_merges = self.config.vocab_size - (self.base_vocab_size + len(self.special_tokens))
        if max_merges <= 0:
            raise ValueError(
                "vocab_size is too small. It must be larger than base bytes + special tokens."
            )

        self.merges.clear()
        self.merges_rank.clear()

        next_token_id = self.base_vocab_size + len(self.special_tokens)
        merge_count = 0
        start_time = time.perf_counter()

        for merge_step in range(max_merges):
            pair_counts: Counter[Tuple[int, int]] = Counter()
            for symbols, freq in word_freqs.items():
                for i in range(len(symbols) - 1):
                    pair_counts[(symbols[i], symbols[i + 1])] += freq

            if not pair_counts:
                break

            best_pair, best_freq = pair_counts.most_common(1)[0]
            if best_freq < self.config.min_pair_freq:
                break

            new_id = next_token_id
            next_token_id += 1

            self.merges[best_pair] = new_id
            self.merges_rank[best_pair] = merge_step
            self.token_to_bytes[new_id] = self.token_to_bytes[best_pair[0]] + self.token_to_bytes[best_pair[1]]

            updated = Counter()
            for symbols, freq in word_freqs.items():
                merged_symbols = self._merge_sequence(symbols, best_pair, new_id)
                updated[merged_symbols] += freq
            word_freqs = updated

            merge_count += 1
            if verbose and (merge_step + 1) % 200 == 0:
                print(f"Merged {merge_step + 1}/{max_merges} pairs...")

        elapsed = time.perf_counter() - start_time
        if verbose:
            print(
                f"Training complete | merges={merge_count} | vocab={self.vocab_size} | time={elapsed:.2f}s"
            )

    def _encode_chunk(self, chunk: str) -> List[int]:
        symbols: List[int] = list(chunk.encode("utf-8"))
        if len(symbols) < 2:
            return symbols

        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")

            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                rank = self.merges_rank.get(pair)
                if rank is not None and rank < best_rank:
                    best_rank = rank
                    best_pair = pair

            if best_pair is None:
                break

            merged_token = self.merges[best_pair]
            out = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best_pair:
                    out.append(merged_token)
                    i += 2
                else:
                    out.append(symbols[i])
                    i += 1
            symbols = out

        return symbols

    def encode(self, text: str, add_bos: bool = False, add_eos: bool = False) -> List[int]:
        if text == "":
            return []

        tokens: List[int] = []
        if add_bos and "<bos>" in self.special_to_id:
            tokens.append(self.special_to_id["<bos>"])

        for chunk in self._pretokenize(text):
            tokens.extend(self._encode_chunk(chunk))

        if add_eos and "<eos>" in self.special_to_id:
            tokens.append(self.special_to_id["<eos>"])

        return tokens

    def decode(self, token_ids: List[int], skip_special_tokens: bool = False) -> str:
        byte_stream = bytearray()
        for tid in token_ids:
            if skip_special_tokens and tid in self.id_to_special:
                continue
            token_bytes = self.token_to_bytes.get(tid)
            if token_bytes is None:
                if "<unk>" in self.special_to_id:
                    token_bytes = self.token_to_bytes[self.special_to_id["<unk>"]]
                else:
                    continue
            byte_stream.extend(token_bytes)

        return bytes(byte_stream).decode("utf-8", errors="replace")

    def save(self, path: str | Path) -> None:
        path = Path(path)
        payload = {
            "config": {
                "vocab_size": self.config.vocab_size,
                "min_pair_freq": self.config.min_pair_freq,
                "special_tokens": list(self.special_tokens),
            },
            "merges": [[a, b, new_id] for (a, b), new_id in self.merges.items()],
        }
        path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, path: str | Path) -> "BytePairTokenizer":
        path = Path(path)
        payload = json.loads(path.read_text(encoding="utf-8"))

        config_raw = payload["config"]
        tokenizer = cls(
            TokenizerConfig(
                vocab_size=config_raw["vocab_size"],
                min_pair_freq=config_raw.get("min_pair_freq", 2),
                special_tokens=tuple(config_raw.get("special_tokens", ["<pad>", "<bos>", "<eos>", "<unk>"])),
            )
        )

        tokenizer.merges.clear()
        tokenizer.merges_rank.clear()
        tokenizer.token_to_bytes = {i: bytes([i]) for i in range(tokenizer.base_vocab_size)}
        tokenizer._init_special_tokens()

        for rank, (a, b, new_id) in enumerate(payload["merges"]):
            pair = (int(a), int(b))
            new_id = int(new_id)
            tokenizer.merges[pair] = new_id
            tokenizer.merges_rank[pair] = rank
            tokenizer.token_to_bytes[new_id] = tokenizer.token_to_bytes[pair[0]] + tokenizer.token_to_bytes[pair[1]]

        return tokenizer

In [9]:
# Train tokenizer on Wizard of Oz
data_path = Path("..") / "wizard_of_oz.txt"
text = data_path.read_text(encoding="utf-8")

tokenizer = BytePairTokenizer(
    TokenizerConfig(
        vocab_size=2000,
        min_pair_freq=2,
        special_tokens=("<pad>", "<bos>", "<eos>", "<unk>"),
    )
)

tokenizer.train(text, verbose=True)
print(f"Final tokenizer vocab size: {tokenizer.vocab_size}")

Merged 200/1740 pairs...
Merged 400/1740 pairs...
Merged 600/1740 pairs...
Merged 800/1740 pairs...
Merged 1000/1740 pairs...
Merged 1200/1740 pairs...
Merged 1400/1740 pairs...
Merged 1600/1740 pairs...
Training complete | merges=1740 | vocab=2000 | time=20.03s
Final tokenizer vocab size: 2000


In [10]:
# Validate encode/decode quality and token compression
sample = "Dorothy lived in the midst of the great Kansas prairies, with Uncle Henry and Aunt Em."
encoded = tokenizer.encode(sample, add_bos=True, add_eos=True)
decoded = tokenizer.decode(encoded)

raw_bytes = len(sample.encode("utf-8"))
token_count = len(encoded)
compression_ratio = raw_bytes / max(token_count, 1)

print("Original:", sample)
print("Decoded :", decoded)
print("Match   :", sample == decoded.replace("<bos>", "").replace("<eos>", ""))
print(f"Bytes={raw_bytes}, Tokens={token_count}, Bytes/Token={compression_ratio:.3f}")
print("First 40 token ids:", encoded[:40])

Original: Dorothy lived in the midst of the great Kansas prairies, with Uncle Henry and Aunt Em.
Decoded : <bos>Dorothy lived in the midst of the great Kansas prairies, with Uncle Henry and Aunt Em.<eos>
Match   : True
Bytes=86, Tokens=42, Bytes/Token=2.048
First 40 token ids: [257, 355, 32, 1556, 32, 264, 32, 261, 32, 1560, 286, 32, 282, 32, 261, 32, 536, 32, 1467, 32, 112, 345, 302, 751, 44, 32, 325, 32, 1290, 32, 1758, 32, 268, 32, 65, 322, 116, 32, 69, 109]


In [11]:
# Save and reload tokenizer
save_path = Path("bpe_tokenizer_wizard.json")
tokenizer.save(save_path)

reloaded = BytePairTokenizer.load(save_path)
encoded_2 = reloaded.encode(sample, add_bos=True, add_eos=True)
decoded_2 = reloaded.decode(encoded_2)

print("Tokenizer saved to:", save_path.resolve())
print("Encoding stable after reload:", encoded == encoded_2)
print("Decoding stable after reload:", decoded == decoded_2)

Tokenizer saved to: C:\Users\Deepesh\Desktop\Mini_Generative_Pretrained_Transformer\Research\bpe_tokenizer_wizard.json
Encoding stable after reload: True
Decoding stable after reload: True
